In [1]:
import json
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)

In [12]:
test_folder_path = Path("./tests")
test = "spectra"

In [14]:
with open(test_folder_path / test / f"{test}_specs.json" ) as f:
    test_specs = json.load(f)

norms_path = test_folder_path / test / f"{test}_norms.csv"
norms = pd.read_csv(norms_path)

In [164]:
def convert_to_matrices(test):
    # get test length
    test_length = test["length"]
    # init matrices to be returned
    df_straight = pd.DataFrame()
    df_reversed = pd.DataFrame()
    # iterate over scales
    for scale in test["scales"]:
        # get scale label, straight and reversed items
        scale_label, straight_items_indices, reversed_items_indices = scale
        # iterate over type of items (either straight or reversed)
        for df, items_indices in [(df_straight, straight_items_indices), (df_reversed, reversed_items_indices)]:
            # init new series with zeroes 
            zeros = pd.Series(np.zeros(test_length))
            # correct items indices, since items are 1-based while matrix indices are 0-based
            matrix_indices = pd.Series(items_indices).sub(1)
            # "switch to 1" scale items in zeros series
            zeros[matrix_indices] = 1
            # add scale items matrix to df
            df[scale_label] = zeros
    return df_straight, df_reversed

def compute_raw_scores(data, test_specs, score_strategy = "sum"):
    
    # init results
    results = pd.DataFrame()
    # clone data
    answers = data.copy()
    # get scales items matrices (two separate matrices for straight and reversed items)
    df_straight, df_reversed = convert_to_matrices(test_specs)
    
    ############################################################
    # straight items
    ############################################################
    
    # compute sum of straight items
    score_straight_sum = np.dot(answers.fillna(0), df_straight)
    # compute mean of straight items
    score_straight_mean = (score_straight_sum / df_straight.sum(axis=0).to_numpy())

    ############################################################
    # reversed items
    ############################################################
    
    # compute amount to use for reversed items
    rev = test_specs["likert"]["min"] + test_specs["likert"]["max"]
    # compute sum of reversed items
    score_reversed_sum = np.dot(rev - answers.fillna(rev), df_reversed)
    # compute mean of reversed items
    with np.errstate(divide='ignore', invalid='ignore'):
        score_reversed_mean = np.true_divide(score_reversed_sum, df_reversed.sum(axis=0).to_numpy())
        score_reversed_mean[score_reversed_mean == np.inf] = 0
        score_reversed_mean = np.nan_to_num(score_reversed_mean)

    #############################################################
    # reversed items
    ############################################################
    
    # compute final results
    results.loc[:, df_straight.columns] = score_straight_sum + score_reversed_sum
    if score_strategy == "mean":
        results.loc[:, df_straight.columns] = score_straight_mean + score_reversed_mean
    # return results
    return results

def compute_standard_score(s, **kwargs):
    # get kwargs
    norms, type = kwargs["norms"],  kwargs["type"]
    # return standard scores
    return norms[norms["scale"].eq(s.name)].take(s)[type].values

In [165]:
# load data
data = pd.read_csv("data.csv")
data.head()

,i1,i2,i3,i4,i5,i6,i7,i8,i9,i10,i11,i12,i13,i14,i15,i16,i17,i18,i19,i20,i21,i22,i23,i24,i25,i26,i27,i28,i29,i30,i31,i32,i33,i34,i35,i36,i37,i38,i39,i40,i41,i42,i43,i44,i45,i46,i47,i48,i49,i50,i51,i52,i53,i54,i55,i56,i57,i58,i59,i60,i61,i62,i63,i64,i65,i66,i67,i68,i69,i70,i71,i72,i73,i74,i75,i76,i77,i78,i79,i80,i81,i82,i83,i84,i85,i86,i87,i88,i89,i90,i91,i92,i93,i94,i95,i96
0,NaN,2,4,2,1,2,3,4,1,3,3,3,1,3,1,3,2,NaN,3,2,1,2,2,3,4,2,3,2,1,3,4,3,4,2,4,3,3,3,3,3,4,1,2,4,3,4,3,2,2,3,1,3,1,2,3,1,4,2,1,2,4,3,2,2,1,4,2,2,1,1,4,3,2,1,3,3,1,3,4,3,2,3,3,4,4,1,2,1,2,2,3,2,2,1,4,2
1,4.0,4,2,3,2,2,3,2,3,4,1,3,3,3,4,3,2,2.0,3,4,3,2,1,4,4,2,2,4,1,1,4,4,3,2,1,1,3,4,1,4,3,4,1,3,1,2,4,2,2,3,2,1,4,3,4,2,1,1,4,4,4,3,3,2,2,3,1,4,1,3,3,2,3,1,3,4,4,3,1,2,4,2,3,2,3,2,3,2,4,3,1,1,1,2,4,2


In [166]:
raw_scores = compute_raw_scores(data, test_specs, score_strategy = "mean")
raw_scores

,dep,anx,soc,pts,alc,agg,anti,drg,psy,par,man,gra,cog,pf,sui,int,ext,ri,gpi
0,1.333333,2.875,2.2,2.166667,3.333333,2.833333,1.5,2.0,3.0,2.666667,2.4,3.166667,2.8,2.125,2.666667,2.2,2.346154,2.826087,2.445946
1,2.333333,2.5,2.4,3.0,2.833333,2.5,3.0,2.666667,2.5,3.166667,2.6,2.666667,2.6,2.625,1.833333,2.56,2.769231,2.73913,2.689189


In [167]:
t_scores = raw_scores.apply(compute_standard_score, norms=norms, type="tscore")
t_scores

,dep,anx,soc,pts,alc,agg,anti,drg,psy,par,man,gra,cog,pf,sui,int,ext,ri,gpi
0,40,40,38,39,43,41,40,44,42,42,40,35,38,14,43,38,41,36,37
1,40,40,38,39,43,41,40,44,42,42,40,35,38,14,43,38,41,36,37
